# Lab 21 — Fine-tuning LLMs · RUN ALL (T4) — Lương Quốc Khánh

Personalized submission runner for **2A202601713**. Chạy từ trên xuống. Runtime → Change runtime type → **T4 GPU** trước khi bắt đầu.

| Ô | Làm gì | Thời gian |
|---|---|---|
| 1 | clone fork + install | ~1–2 phút |
| 2 | smoke: import + unit test | ~30 giây |
| 3 | **full core pipeline NB1 → NB5** | ~100–130 phút |
| 4 | xem gatekeeper | ~10 giây |
| 5 | bổ sung qualitative (b), sinh REPORT, verify, tải bundle | ~2–5 phút |

> Bài nộp mặc định dùng **full eval** (`EVAL_LIMIT=""`) và **2 epochs**. Không đổi sang smoke mode khi nộp.


In [ ]:
# @title 1. Setup — clone fork của Khánh + install
import os, subprocess, sys

REPO = "https://github.com/QuocKhanhLuong/Day21-Track3-Finetuning-Lab-2A202601713-LuongQuocKhanh.git"
LOCAL_DIR = "/content/Day21-Track3-Finetuning-Lab"
if not os.path.exists(LOCAL_DIR):
    subprocess.run(["git", "clone", "-q", REPO, LOCAL_DIR], check=True)
os.chdir(LOCAL_DIR)
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests
!python scripts/verify.py --smoke


In [ ]:
# @title 3. FULL submission pipeline — NB1 → NB5
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""           # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
os.environ.pop("EPOCHS", None)  # submission default = 2 epochs
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")
!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + raw results (REPORT chưa sinh ở bước này)
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null
print("\nNOTE: REPORT template còn placeholder ở bước 4 là bình thường; chạy ô 5 để finalize.")


In [ ]:
# @title 5. Finalize REPORT + verify + tải submission bundle
# Chấm optimized baseline (b) trên đúng 5 case qualitative được chọn, rồi tự sinh report từ results/.
!python scripts/score_qualitative_b.py
!python scripts/finalize_submission.py --name "Lương Quốc Khánh" --student-id "2A202601713" --gpu "Tesla T4 (Colab)"
!python scripts/verify.py

import pathlib, shutil
from google.colab import files
bundle = pathlib.Path("/content/lab21-2A202601713")
if bundle.exists():
    shutil.rmtree(bundle)
(bundle / "submission").mkdir(parents=True)
(bundle / "adapters").mkdir(parents=True)
shutil.copytree("results", bundle / "results")
shutil.copytree("adapters/correct", bundle / "adapters/correct")
shutil.copy2("submission/REPORT.md", bundle / "submission/REPORT.md")
archive = shutil.make_archive("/content/lab21-2A202601713-artifacts", "zip", root_dir=bundle)
print("bundle:", archive)
files.download(archive)
